## 1. Read the raw energy file

The source contains Unix timestamps in seconds and two power measurements. 

In [9]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

input_file = root / "data" / "raw data" / "energy-data.csv"
output_file = root / "data" / "processed data" / "energy_2025.csv"

raw = pd.read_csv(input_file, encoding="utf-8-sig",
                  usecols=["unix_seconds", "load", "solar"])
print(f"Rows read: {len(raw):,}")
display(raw.head(3))

Rows read: 24,752


,unix_seconds,load,solar
0,1704063600,8983.0,0.0
1,1704067200,8598.7,0.0
2,1704070800,8794.0,0.0


## 2. Convert Unix seconds to Swiss local time

Unix seconds identify an instant relative to the UTC epoch. 

We first parse them in UTC, then convert the same instants to Europe/Zurich. 

This assigns the correct winter or summer offset to each row. 

During the autumn clock change, the two occurrences of 02:00 remain distinguishable.

In [11]:
utc_time = pd.to_datetime(energy["unix_seconds"], unit="s", utc=True)
energy["timestamp"] = utc_time.dt.tz_convert("Europe/Zurich")

display(energy[["unix_seconds", "timestamp"]].iloc[[0, -1]])

,unix_seconds,timestamp
0,1704063600,2024-01-01 00:00:00+01:00
24751,1793167200,2026-10-28 07:00:00+01:00


## 3. Select the 2025 study period

We select rows by their year in Zurich. 

For example, midnight on 1 January 2025 in Zurich is still 31 December 2024 in UTC. 

Applying the year filter after conversion keeps that first local hour. 

Sorting gives us a chronological series for the merge.

In [12]:
columns = ["timestamp", "load", "solar"]
energy_2025 = (
    energy.loc[energy["timestamp"].dt.year == 2025, columns]
    .sort_values("timestamp")
    .reset_index(drop=True)
)
print(f"Rows in 2025: {len(energy_2025):,}")
display(energy_2025.head(2), energy_2025.tail(2))

Rows in 2025: 8,760


,timestamp,load,solar
0,2025-01-01 00:00:00+01:00,7880.5,0.0
1,2025-01-01 01:00:00+01:00,8432.4,0.0


,timestamp,load,solar
8758,2025-12-31 22:00:00+01:00,8209.3,0.0
8759,2025-12-31 23:00:00+01:00,8305.2,0.0


## 4. Check hourly coverage and measurements

For the hourly series we expect 8,760 distinct instants in 2025. 

We compare the actual timestamps with a timezone-aware hourly calendar. 

This detects missing hours, duplicate timestamps and observations between hours, even if their counts happen to cancel out. 

We also check that load and solar are numeric and complete. 

Zero solar generation is a valid measurement.

In [13]:
expected = pd.date_range("2025-01-01", "2026-01-01", freq="h",
                         tz="Europe/Zurich", inclusive="left")
actual = pd.DatetimeIndex(energy_2025["timestamp"])

for column in ["load", "solar"]:
    energy_2025[column] = pd.to_numeric(energy_2025[column], errors="raise")

checks = pd.Series({
    "Missing expected hours": len(expected.difference(actual)),
    "Unexpected timestamps": len(actual.difference(expected)),
    "Duplicate timestamps": int(actual.duplicated().sum()),
    "Missing load values": int(energy_2025["load"].isna().sum()),
    "Missing solar values": int(energy_2025["solar"].isna().sum()),
})
display(checks.to_frame("Count"))
if checks.any():
    raise ValueError("Review the non-zero counts before exporting.")

,Count
Missing expected hours,0
Unexpected timestamps,0
Duplicate timestamps,0
Missing load values,0
Missing solar values,0


## 5. Export the prepared energy data

We write the three columns to a separate CSV. 

We use ISO 8601 timestamps with seconds and an explicit offset, for example `2025-01-01T00:00:00+01:00`.

The timezone-aware values stay available in the notebook. 

When reading the CSV for the merge, parse both datasets’ offset-aware timestamp columns with `pd.to_datetime(..., utc=True)` so the join matches the same instant across locations.

In [16]:
export = energy_2025.copy()
export["timestamp"] = export["timestamp"].map(lambda t: t.isoformat(timespec="seconds"))
output_file.parent.mkdir(parents=True, exist_ok=True)
export.to_csv(output_file, index=False)

print(f"Saved {len(export):,} hourly records to {output_file}")
display(export.head(3))

Saved 8,760 hourly records to /home/simon/dspro01_hs26_group-XX-/data/processed data/energy_2025.csv


,timestamp,load,solar
0,2025-01-01T00:00:00+01:00,7880.5,0.0
1,2025-01-01T01:00:00+01:00,8432.4,0.0
2,2025-01-01T02:00:00+01:00,8373.9,0.0
